# Biomarker Analysis Pipeline

Runs all cohort combinations:
- **Cohorts**: `all_ICI` (any-line ICI vs never-ICI), `first_line` (first-line ICI vs never-ICI)

Each run produces 3 sensitivity specs internally: stabilized ATE, stabilized ATT, and unweighted (noIPTW).

### Stages
1. **Propensity scores** — `ICI_LRs_all_ICI.py`, `ICI_LRs_first_line.py`
2. **IPTW datasets** — `generate_IPTW_df.py --cohort {cohort}`
3. **Cox models** — `run_IPTW_analysis.py --cohort {cohort}`

In [7]:
import subprocess
import sys
import os

SCRIPT_DIR = os.path.dirname(os.path.abspath('__file__'))
COHORTS = ['all_ICI', 'first_line']

def run_and_stream(label, cmd):
    """Run a command and stream its output inline."""
    print(f"\n--- {label} ---")
    result = subprocess.run(cmd, cwd=SCRIPT_DIR,
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                            universal_newlines=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"FAILED (exit code {result.returncode})")
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"{label} failed")
    print(f"Done: {label}")

## Stage 1: Propensity Score Generation

In [2]:
run_and_stream('all_ICI propensity', [sys.executable, 'ICI_LRs_all_ICI.py'])
run_and_stream('first_line propensity', [sys.executable, 'ICI_LRs_first_line.py']) 


--- all_ICI propensity ---
ICI (IO_START ∩ cohort):      6699
Never ICI (treated, no ICI):  8741
IO_START not in cohort:       2223
Figure(2000x500)

Done: all_ICI propensity

--- first_line propensity ---
First-line ICI (IO_START ∩ first-line ∩ cohort): 2801
ICI in IO_START but not first-line:              3898
Never ICI (treated, no ICI):                     8741
IO_START not in cohort:                          2223
Figure(2000x500)

Done: first_line propensity


## Stage 2: IPTW Dataset Generation

In [3]:
for cohort in COHORTS:
    run_and_stream(f'{cohort} IPTW dataset',
                   [sys.executable, 'generate_IPTW_df.py', '--cohort', cohort])


--- all_ICI IPTW dataset ---
[generate_IPTW_df] Cohort: all_ICI
[generate_IPTW_df] Propensity path: /data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/treatment_prediction/all_ICI_propensity/w_30_day_buffer/
[generate_IPTW_df] Prediction data path: /data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/treatment_prediction/all_ICI_prediction_data/
[generate_IPTW_df] Saved 7512 patients to /data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/biomarker_analysis/IPTW_ICI_interaction_runs_df_all_ICI.csv

Done: all_ICI IPTW dataset

--- first_line IPTW dataset ---
[generate_IPTW_df] Cohort: first_line
[generate_IPTW_df] Propensity path: /data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/treatment_prediction/first_line_ICI_propensity/w_30_day_buffer/
[generate_IPTW_df] Prediction data path: /data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/treatment_prediction/first_line_ICI_prediction_data/
[generate_IPTW_df] Saved 5829 patients t

In [18]:
first_line_df = pd.read_csv('/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/biomarker_analysis/IPTW_ICI_interaction_runs_df_first_line.csv')
first_line_df

,DFCI_MRN,tt_death,death,GENDER,AGE_AT_TREATMENTSTART,ABCB11_AMP,ABL1_AMP,ACVR1_AMP,AKT1_AMP,AKT2_AMP,...,CANCER_TYPE_PROSTATE,CANCER_TYPE_SKIN,CANCER_TYPE_SOFT_TISSUE,CANCER_TYPE_STOMACH,CANCER_TYPE_UTERUS,PANEL_VERSION_2.0,PANEL_VERSION_3.0,PANEL_VERSION_3.1,PX_on_ICI,ICI_prediction
0,103663,48.0,1,1.0,68,0,0,0,0,0,...,False,False,False,False,False,False,True,False,0,0.002875
1,107085,1590.0,1,1.0,61,0,0,0,0,0,...,False,False,False,True,False,False,False,True,0,0.691690
2,107217,75.0,1,1.0,70,0,0,0,0,0,...,False,False,True,False,False,False,True,False,0,0.639955
3,108952,26.0,1,1.0,56,0,0,0,0,0,...,False,False,False,False,False,False,True,False,1,0.200159
4,110390,1763.0,0,0.0,60,0,0,0,0,0,...,False,False,False,False,False,False,False,True,0,0.191801
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5824,1126200,636.0,0,1.0,67,0,0,0,0,1,...,False,False,False,False,True,False,False,True,0,0.003436
5825,1126539,666.0,0,0.0,51,0,0,0,0,0,...,False,False,False,False,False,False,False,True,0,0.036386
5826,1127709,536.0,1,1.0,75,0,0,0,0,0,...,False,False,False,False,False,False,False,True,0,0.999908
5827,1130467,127.0,1,1.0,49,0,0,0,0,0,...,False,False,False,False,False,False,False,True,1,0.008410


## Stage 3: IPTW Cox Model Analysis

In [8]:
for cohort in COHORTS:
    run_and_stream(f'{cohort} Cox models',
                   [sys.executable, 'run_IPTW_analysis.py', '--cohort', cohort])


--- all_ICI Cox models ---
[run_IPTW_analysis] Cohort: all_ICI
[run_IPTW_analysis] Output: /data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/biomarker_analysis/IPTW_runs_all_ICI/
[pan_cancer] Merged 2 rare cancer types into CANCER_TYPE_OTHER: CANCER_TYPE_LYMPHOMA, CANCER_TYPE_MYELOMA
[pan_cancer] ATE: N treated=2926, N control=4411 | ESS treated=528, ESS control=1015
[pan_cancer] ATT: N treated=2926, N control=4411 | ESS treated=2926, ESS control=724
[SKIN] Recalibrated propensity scores within subset (mean 0.754 -> 0.936)
[SKIN] ATE: N treated=378, N control=29 | ESS treated=378, ESS control=29
[SKIN] ATT: N treated=378, N control=29 | ESS treated=378, ESS control=29
[LUNG] Recalibrated propensity scores within subset (mean 0.579 -> 0.603)
[LUNG] ATE: N treated=762, N control=503 | ESS treated=759, ESS control=499
[LUNG] ATT: N treated=762, N control=503 | ESS treated=762, ESS control=492

Done: all_ICI Cox models

--- first_line Cox models ---
[run_IPTW_analysis] Cohor

In [14]:
output_path = '/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/biomarker_analysis/'
all_ICI_path = os.path.join(output_path, 'IPTW_runs_all_ICI/')
first_line_path = os.path.join(output_path, 'IPTW_runs_first_line/')
compiled_df_path = os.path.join(output_path, 'compiled_IPTW_results/')
os.makedirs(compiled_df_path, exist_ok=True)

import pandas as pd
for run_id in ['all_ICI', 'first_line']:
    run_path = os.path.join(output_path, f'IPTW_runs_{run_id}/')
    
    for IPTW_type in ['ATT', 'ATE']:
        
        pc_df = pd.read_csv(os.path.join(run_path, f'pan_cancer_{IPTW_type}_ICI_predictive_markers.csv'))
        lung_df = pd.read_csv(os.path.join(run_path, f'LUNG_{IPTW_type}_ICI_predictive_markers.csv'))
        skin_df = pd.read_csv(os.path.join(run_path, f'SKIN_{IPTW_type}_ICI_predictive_markers.csv'))

        cols_to_select = ['marker', 'beta_markerxICI', 'p_markerxICI', 'FDR_markerxICI', 'classifier']

        pc_hits = pc_df.loc[pc_df['significant_predictive'], cols_to_select]
        pc_hits['cancer_specificity'] = 'pan_cancer'

        lung_hits = lung_df.loc[lung_df['significant_predictive'], cols_to_select]
        lung_hits['cancer_specificity'] = 'lung'

        skin_hits = skin_df.loc[skin_df['significant_predictive'], cols_to_select]
        skin_hits['cancer_specificity'] = 'skin'

        complete_hits = pd.concat([pc_hits, lung_hits, skin_hits])

        complete_hits.to_csv(os.path.join(compiled_df_path, f'{run_id}_{IPTW_type}_compiled_hits.csv'),index=False)

In [15]:
os.listdir(compiled_df_path)

['all_ICI_ATT_compiled_hits.csv',
 'all_ICI_ATE_compiled_hits.csv',
 'first_line_ATT_compiled_hits.csv',
 'first_line_ATE_compiled_hits.csv']

In [13]:
pc_df = pd.read_csv(os.path.join(all_ICI_path, 'pan_cancer_ATT_ICI_predictive_markers.csv'))
lung_df = pd.read_csv(os.path.join(all_ICI_path, 'LUNG_ATT_ICI_predictive_markers.csv'))
skin_df = pd.read_csv(os.path.join(all_ICI_path, 'SKIN_ATT_ICI_predictive_markers.csv'))

cols_to_select = ['marker', 'beta_markerxICI', 'p_markerxICI', 'FDR_markerxICI', 'classifier']

pc_hits = pc_df.loc[pc_df['significant_predictive'], cols_to_select]
pc_hits['cancer_specificity'] = 'pan_cancer'

lung_hits = lung_df.loc[lung_df['significant_predictive'], cols_to_select]
lung_hits['cancer_specificity'] = 'lung'

skin_hits = skin_df.loc[skin_df['significant_predictive'], cols_to_select]
skin_hits['cancer_specificity'] = 'skin'

complete_hits = pd.concat([pc_hits, lung_hits, skin_hits])

complete_hits.to_csv(os.path.join(all_ICI_path, 'all_ICI_compiled_hits.csv'),index=False)
complete_hits

,marker,beta_markerxICI,p_markerxICI,FDR_markerxICI,classifier,cancer_specificity
42,BRD4_AMP,-0.590430,5.858425e-05,0.027828,predictive_ICI_benefit,pan_cancer
111,EGFR_AMP,-0.385723,1.884064e-04,0.034320,predictive_ICI_benefit,pan_cancer
129,ERCC6_AMP,-0.876638,6.785958e-04,0.046649,predictive_ICI_benefit,pan_cancer
177,GLI2_AMP,-0.714837,2.167607e-04,0.034320,predictive_ICI_benefit,pan_cancer
281,NOTCH3_AMP,-0.571370,1.043659e-03,0.049853,predictive_ICI_benefit,pan_cancer
311,PNKP_AMP,-0.513495,9.624636e-04,0.049853,predictive_ICI_benefit,pan_cancer
314,POLD1_AMP,-0.576100,4.500367e-04,0.046649,predictive_ICI_benefit,pan_cancer
321,PPP2R1A_AMP,-0.544936,6.874623e-04,0.046649,predictive_ICI_benefit,pan_cancer
401,SMARCA4_AMP,-0.504093,1.049543e-03,0.049853,predictive_ICI_benefit,pan_cancer
472,ZNF708_AMP,-1.025722,6.064993e-04,0.046649,predictive_ICI_benefit,pan_cancer
